# RAG Pipeline — Baseline Qwen (Error-Based Retrieval + Summarised Hints)

This notebook builds an **error-driven RAG** pipeline for the **baseline** (non-fine-tuned) Qwen 2.5 Coder 7B model.

### Error Categories & Handling Strategy

| Category | Count | Strategy |
|----------|-------|----------|
| **SyntaxError** | 4 | No traceback available. Prompt: *"fix the syntax error in this code"* + code only |
| **NameError** | ~2 | Traceback available but RAG is unhelpful (typos). Prompt: code + traceback only |
| **Other runtime** | ~14 | Full RAG pipeline: retrieve → rerank → summarise → code + traceback + bullet hints |
| **Timeout** | 1 | No traceback. Prompt: code only (best-effort) |

### Workflow

1. **Load smoke_report_A** — Read `smoke_report_A.json`, collect **all** `ok=false` entries (21 total)  
2. **Classify error** — Categorise each entry as `syntax_error`, `name_error`, `timeout`, or `rag_error`  
3. **Read failed prediction code** — Load `prediction.py` for each failed sample  
4. **Branch by category:**
   - *syntax / name / timeout* → Prompt code-fixer directly (no RAG)
   - *rag_error* → Retrieve 10 docs → rerank 3 → summarise into bullets → prompt code-fixer  
5. **Parse & Save** — Extract corrected code, save all artifacts

### Models

| Role | Model | Why |
|------|-------|-----|
| **Code fixer** | `qwen/qwen-2.5-coder-7b-instruct` | Same model used for baseline pre-test |
| **Doc summariser** | `mistralai/mistral-7b-instruct:free` | Small, fast, strong at summarisation, free tier |
| **Bi-encoder** | `BAAI/bge-base-en-v1.5` | Fast retrieval from ChromaDB |
| **Cross-encoder** | `BAAI/bge-reranker-base` | Precise reranking |

## 1 — Imports

In [2]:
import json, os, re, time, random
import importlib.metadata
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter
from dotenv import load_dotenv

from tqdm.auto import tqdm
from datasets import Dataset

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import CrossEncoder
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

print("All imports successful.")

All imports successful.


In [3]:
import truststore
truststore.inject_into_ssl()
print("\u2713 Injected OS certificate store (truststore) for SSL verification.")

✓ Injected OS certificate store (truststore) for SSL verification.


## 2 — Configuration

In [6]:
#    Paths                                                                        
DATA_PATH       = Path("../Datasets/final_dataset.json").resolve()
CHROMA_DIR      = str(Path("../VectorDB/chroma_library_docs").resolve())
OUT_DIR         = Path("RAG_outputs/RAG_with_Baseline").resolve()
COLLECTION_NAME = "library_docs"

#    Smoke report (source of failed samples + tracebacks)                
SMOKE_REPORT_PATH = Path(
    r"../Fine-Tuning/Qwen/PreTest_Results/preTest_runtime_A/smoke_report_A.json"
).resolve()

#    OpenRouter / LLMs                                                               
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Code-fixer model (same baseline model used in Pre_Test)
CODER_MODEL_ID    = "qwen/qwen2.5-coder-7b-instruct"
CODER_MAX_TOKENS  = 8192

# Summarisation model (small, fast, strong at distilling docs into bullets)
SUMMARIZER_MODEL_ID   = "openai/gpt-oss-20b:free"
SUMMARIZER_MAX_TOKENS = 512

# Dataset split (to match the eval set & get ground truth) 
SEED      = 42
TEST_SIZE = 0.15  # \u2192 88 eval samples

# RAG settings                                                                  
N_RETRIEVE          = 10    # candidates per library from bi-encoder
N_RERANK            = 3     # top-k after cross-encoder reranking
MAX_QUERY           = 500   # max chars of error text used as RAG query
MAX_CTX_CHARS       = 3000  # max total chars of raw RAG context (before summarisation)
MIN_RERANKER_SCORE  = 0.0   # score gate for cross-encoder

#    Reranker                                                                      
RERANKER_MODEL = "BAAI/bge-reranker-base"

print(f"Dataset         : {DATA_PATH}")
print(f"Smoke report    : {SMOKE_REPORT_PATH}")
print(f"ChromaDB        : {CHROMA_DIR}")
print(f"Output dir      : {OUT_DIR}")
print(f"Code-fixer      : {CODER_MODEL_ID}")
print(f"Summariser      : {SUMMARIZER_MODEL_ID}")
print(f"Retrieve        : {N_RETRIEVE}/lib \u2192 rerank \u2192 top-{N_RERANK} (score>{MIN_RERANKER_SCORE})")
print(f"Max RAG ctx     : {MAX_CTX_CHARS} chars (before summarisation)")

Dataset         : C:\Users\hbahmanyar\MentorApp\Datasets\final_dataset.json
Smoke report    : C:\Users\hbahmanyar\MentorApp\Fine-Tuning\Qwen\PreTest_Results\preTest_runtime_A\smoke_report_A.json
ChromaDB        : C:\Users\hbahmanyar\MentorApp\VectorDB\chroma_library_docs
Output dir      : C:\Users\hbahmanyar\MentorApp\RAG_Pipelines\RAG_outputs\RAG_with_Baseline
Code-fixer      : qwen/qwen2.5-coder-7b-instruct
Summariser      : openai/gpt-oss-20b:free
Retrieve        : 10/lib → rerank → top-3 (score>0.0)
Max RAG ctx     : 3000 chars (before summarisation)


## 3 — Load Smoke Report, Classify Errors & Attach Ground Truth

Load `smoke_report_A.json` → keep **all** `ok=false` entries (21 total).  
Classify each into: `syntax_error` | `name_error` | `timeout` | `rag_error`.  
Also load the dataset to get ground-truth `correct_code` for each sample.

In [7]:
random.seed(SEED)

# ── Load ground-truth dataset + eval split ────────────────────────────────────
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)
split = dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)
eval_dataset = split["test"]
eval_examples = [eval_dataset[i] for i in range(len(eval_dataset))]

print(f"Total dataset : {len(data)}")
print(f"Eval split    : {len(eval_examples)}")

# ── Load smoke_report_A and keep ALL failed entries ───────────────────────────
with open(SMOKE_REPORT_PATH, "r", encoding="utf-8") as f:
    smoke_data = json.load(f)

# Keep every ok=false entry (syntax, name, runtime, timeout — all 21)
failed_entries = [e for e in smoke_data if not e["ok"]]

print(f"\nSmoke report    : {len(smoke_data)} total entries")
print(f"Failed (ok=false) : {len(failed_entries)}")


# ── Classify each failed entry ────────────────────────────────────────────────
def classify_error(entry: dict) -> str:
    """
    Classify a failed smoke_report entry into one of:
      syntax_error  — skipped due to syntax failure (no stderr)
      name_error    — NameError in traceback (typos, no RAG benefit)
      timeout       — timed out (no traceback)
      rag_error     — other runtime errors (RAG may help)
    """
    if entry.get("skipped"):
        return "syntax_error"
    stderr = entry.get("stderr_tail", "")
    if entry.get("returncode") is None:
        return "timeout"
    if "NameError" in stderr:
        return "name_error"
    return "rag_error"


for entry in failed_entries:
    entry["error_category"] = classify_error(entry)


# ── For each failed entry, load prediction.py and attach ground truth ─────────
def is_valid_code(code: str) -> bool:
    """Check if code is valid (not a placeholder or too short)."""
    if not code or len(code.strip()) < 50:
        return False
    # Detect common placeholder patterns
    placeholders = [
        "...full corrected python code...",
        "(python code only)",
        "...one short bug type...",
    ]
    for ph in placeholders:
        if ph in code:
            return False
    return True


for entry in failed_entries:
    pred_path = Path(entry["file"])
    pretest_code = pred_path.read_text(encoding="utf-8") if pred_path.exists() else ""
    
    eval_idx = entry["idx"] - 1  # smoke idx is 1-based
    entry["eval_sample"] = eval_examples[eval_idx]
    
    # Use Pre_Test output if valid, otherwise fallback to dataset's incorrect_code
    if is_valid_code(pretest_code):
        entry["prediction_code"] = pretest_code
        entry["code_source"] = "pretest"
    else:
        fallback_code = entry["eval_sample"].get("incorrect_code", "")
        if is_valid_code(fallback_code):
            entry["prediction_code"] = fallback_code
            entry["code_source"] = "dataset_fallback"
        else:
            entry["prediction_code"] = ""
            entry["code_source"] = "invalid"
    
    entry["valid_input"] = bool(entry["prediction_code"].strip())


# ── Print summary by category ────────────────────────────────────────────────
cat_counts = Counter(e["error_category"] for e in failed_entries)
valid_count = sum(1 for e in failed_entries if e["valid_input"])
invalid_count = len(failed_entries) - valid_count
pretest_count = sum(1 for e in failed_entries if e.get("code_source") == "pretest")
fallback_count = sum(1 for e in failed_entries if e.get("code_source") == "dataset_fallback")

print(f"\nCategories:")
for cat in ["syntax_error", "name_error", "rag_error", "timeout"]:
    print(f"  {cat:15s}: {cat_counts.get(cat, 0)}")

print(f"\nInput validation:")
print(f"  Valid code      : {valid_count}")
print(f"    from Pre_Test : {pretest_count}")
print(f"    from dataset  : {fallback_count} (fallback)")
print(f"  Invalid code    : {invalid_count} (will be skipped)")

print(f"\nAll {len(failed_entries)} failed entries:")
for e in failed_entries:
    stderr = e.get("stderr_tail", "")
    err_lines = [l.strip() for l in stderr.split("\n") if l.strip()]
    last_err = err_lines[-1] if err_lines else e.get("reason", "?")
    src = e.get("code_source", "?")
    valid_flag = f"✓ ({src})" if e["valid_input"] else "✗ INVALID"
    print(
        f"  idx={e['idx']:2d}  [{e['error_category']:13s}]  {valid_flag:18s}  "
        f"{e['eval_sample']['title'][:32]:<32s}  "
        f"→ {last_err[:50]}"
    )

Total dataset : 582
Eval split    : 88

Smoke report    : 88 total entries
Failed (ok=false) : 21

Categories:
  syntax_error   : 4
  name_error     : 2
  rag_error      : 14
  timeout        : 1

Input validation:
  Valid code      : 21
    from Pre_Test : 21
    from dataset  : 0 (fallback)
  Invalid code    : 0 (will be skipped)

All 21 failed entries:
  idx= 2  [syntax_error ]  ✓ (pretest)         CIFAR-10 Deep CNN Data Augmentat  → syntax_failed
  idx=11  [syntax_error ]  ✓ (pretest)         MNIST Digit Distribution Check    → syntax_failed
  idx=13  [syntax_error ]  ✓ (pretest)         Fashion MNIST CNN Early Stopping  → syntax_failed
  idx=14  [name_error   ]  ✓ (pretest)         Digits Classification using Conv  → NameError: name 'encodr' is not de
  idx=18  [rag_error    ]  ✓ (pretest)         Reuters News Topic Classificatio  → ValueError: Received an invalid va
  idx=27  [rag_error    ]  ✓ (pretest)         Titanic Survival ROC Curve        → LogisticRegression does not acce

## 4 — Initialize RAG Retriever, Reranker & Venv Version Matching

- **Bi-encoder** — `BAAI/bge-base-en-v1.5` for fast initial retrieval from ChromaDB  
- **Cross-encoder** — `BAAI/bge-reranker-base` for precise reranking  
- **Version matching** — Match installed venv versions to ChromaDB metadata for scoped retrieval

In [8]:
# Embedding function (must match ingestion) 
embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-base-en-v1.5",
    device="cpu",
    normalize_embeddings=True,
)

# Connect to ChromaDB 
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

# Initialize Reranker (cross-encoder) 
reranker = CrossEncoder(RERANKER_MODEL, max_length=512)

print(f"Collection      : {COLLECTION_NAME}")
print(f"Documents       : {collection.count()}")
print(f"Embedding model : BAAI/bge-base-en-v1.5")
print(f"Reranker model  : {RERANKER_MODEL}")

# VENV VERSION INTROSPECTION + CHROMA VERSION MATCHING
_ALL_INSTALLED = {
    d.metadata["Name"].lower(): d.version
    for d in importlib.metadata.distributions()
}

_TRACKED_LIBS = [
    "numpy", "pandas", "scipy", "matplotlib", "seaborn", "scikit-learn",
    "torch", "torchvision", "torchaudio", "tensorflow", "opencv-python",
    "scikit-image", "xgboost", "lightgbm", "catboost", "transformers",
    "datasets", "accelerate", "sentencepiece", "langchain", "pillow",
    "pyarrow", "openpyxl", "requests", "httpx", "tqdm", "statsmodels",
]

VENV_VERSIONS: dict[str, str] = {
    lib: _ALL_INSTALLED[lib]
    for lib in _TRACKED_LIBS
    if lib in _ALL_INSTALLED
}

print(f"\nInstalled library versions ({len(VENV_VERSIONS)} tracked):")
for lib, ver in sorted(VENV_VERSIONS.items()):
    print(f"  {lib:20s}  {ver}")

# Get available versions per library in ChromaDB 
_all_meta = collection.get(include=["metadatas"])["metadatas"]
_CHROMA_VERSIONS: dict[str, set[str]] = {}
for m in _all_meta:
    _CHROMA_VERSIONS.setdefault(m["library"], set()).add(m["version"])

print(f"\nChromaDB versions per library:")
for lib in sorted(_CHROMA_VERSIONS):
    print(f"  {lib:20s}  {sorted(_CHROMA_VERSIONS[lib])}")


# Version matching logic
def _parse_version(v: str) -> tuple:
    parts = []
    for p in re.split(r"[.\-]", v):
        try:
            parts.append(int(p))
        except ValueError:
            break
    return tuple(parts) if parts else (0,)


def best_chroma_version(library: str, installed_version: str) -> str | None:
    available = _CHROMA_VERSIONS.get(library)
    if not available:
        return None
    inst_parsed = _parse_version(installed_version)
    if installed_version in available:
        return installed_version
    inst_prefix = ".".join(str(x) for x in inst_parsed[:2])
    for av in available:
        if av == inst_prefix or av.startswith(inst_prefix + "."):
            return av
    numeric = [
        (av, _parse_version(av))
        for av in available
        if av != "latest" and not av.endswith("x")
    ]
    candidates = [(av, parsed) for av, parsed in numeric if parsed <= inst_parsed]
    if candidates:
        return max(candidates, key=lambda x: x[1])[0]
    if "latest" in available:
        return "latest"
    wildcard = [av for av in available if av.endswith("x")]
    if wildcard:
        return wildcard[0]
    if numeric:
        return min(
            numeric,
            key=lambda x: abs(sum(a - b for a, b in zip(x[1], inst_parsed))),
        )[0]
    return None


VENV_TO_CHROMA: dict[str, str | None] = {
    lib: best_chroma_version(lib, ver) for lib, ver in VENV_VERSIONS.items()
}

print(f"\nVersion mapping (venv \u2192 ChromaDB):")
for lib in sorted(VENV_TO_CHROMA):
    inst = VENV_VERSIONS[lib]
    chroma = VENV_TO_CHROMA[lib]
    flag = "\u2713" if chroma else "\u2717 (no docs)"
    print(f"  {lib:20s}  {inst:12s} \u2192 {str(chroma):12s}  {flag}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 693.13it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 623.95it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection      : library_docs
Documents       : 647
Embedding model : BAAI/bge-base-en-v1.5
Reranker model  : BAAI/bge-reranker-base

Installed library versions (27 tracked):
  accelerate            1.12.0
  catboost              1.2.8
  datasets              4.5.0
  httpx                 0.28.1
  langchain             1.2.10
  lightgbm              4.6.0
  matplotlib            3.10.8
  numpy                 2.4.2
  opencv-python         4.13.0.92
  openpyxl              3.1.5
  pandas                3.0.0
  pillow                12.1.1
  pyarrow               23.0.1
  requests              2.32.5
  scikit-image          0.26.0
  scikit-learn          1.8.0
  scipy                 1.17.0
  seaborn               0.13.2
  sentencepiece         0.2.1
  statsmodels           0.14.6
  tensorflow            2.20.0
  torch                 2.10.0
  torchaudio            2.10.0
  torchvision           0.25.0
  tqdm                  4.67.3
  transformers          5.2.0
  xgboost               

## 5 — Initialize LLM Clients

Two separate LLM clients:
- **Code-fixer** (`Qwen 2.5 Coder 7B`) — generates corrected code
- **Summariser** (`Mistral 7B`) — condenses RAG docs into concise bullet-point hints

In [17]:
_OPENROUTER_HEADERS = {
    "HTTP-Referer": "http://localhost",
    "X-Title": "MentorApp-RAG-Baseline",
}

# Code-fixer LLM 
llm_coder = ChatOpenAI(
    model=CODER_MODEL_ID,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.0,
    max_tokens=CODER_MAX_TOKENS,
    default_headers=_OPENROUTER_HEADERS,
)

# Summariser LLM 
llm_summarizer = ChatOpenAI(
    model=SUMMARIZER_MODEL_ID,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.0,
    max_tokens=SUMMARIZER_MAX_TOKENS,
    default_headers=_OPENROUTER_HEADERS,
)


def _call_llm(llm, messages, retries=6, min_backoff=1.0, max_backoff=20.0):
    """Call an LLM with retry / exponential backoff.  Returns (content, latency_s)."""
    t0 = time.perf_counter()
    last_err = None
    for attempt in range(retries):
        try:
            resp = llm.invoke(messages)
            return resp.content, time.perf_counter() - t0
        except Exception as e:
            last_err = e
            sleep_s = min(max_backoff, min_backoff * (2 ** attempt)) + random.random()
            print(f"  [WARN] attempt {attempt+1}/{retries} failed ({type(e).__name__}): {e}")
            print(f"         retrying in {sleep_s:.1f}s ...")
            time.sleep(sleep_s)
    raise last_err


print(f"Code-fixer  ready : {CODER_MODEL_ID}")
print(f"Summariser  ready : {SUMMARIZER_MODEL_ID}")

Code-fixer  ready : qwen/qwen2.5-coder-7b-instruct
Summariser  ready : openai/gpt-oss-20b:free


## 6 — Library Detection, RAG Retrieval, Summarisation & Prompt Helpers

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# LIBRARY DETECTION
# ══════════════════════════════════════════════════════════════════════════════

_IMPORT_TO_LIB = {
    "sklearn": "scikit-learn", "skimage": "scikit-image",
    "cv2": "opencv-python", "PIL": "pillow", "Pillow": "pillow",
    "np": "numpy", "pd": "pandas", "tf": "tensorflow",
    "plt": "matplotlib", "sns": "seaborn",
    "xgb": "xgboost", "lgb": "lightgbm", "catboost": "catboost",
    "torch": "torch", "torchvision": "torchvision", "torchaudio": "torchaudio",
    "transformers": "transformers", "datasets": "datasets",
    "accelerate": "accelerate", "sentencepiece": "sentencepiece",
    "langchain": "langchain", "scipy": "scipy",
    "pyarrow": "pyarrow", "openpyxl": "openpyxl",
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "seaborn": "seaborn", "tensorflow": "tensorflow",
    "requests": "requests", "httpx": "httpx", "tqdm": "tqdm",
    "statsmodels": "statsmodels", "keras": "tensorflow",
}


def detect_libraries(code: str) -> list[str]:
    """Extract library names from import statements in the code."""
    libs = set()
    for m in re.finditer(r"(?:from|import)\s+([\w.]+)", code):
        top_module = m.group(1).split(".")[0]
        lib = _IMPORT_TO_LIB.get(top_module, top_module)
        libs.add(lib)
    return sorted(libs)


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# TRACEBACK EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════

# Regex to strip ANSI escape codes that appear in stderr
_ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")


def clean_ansi(text: str) -> str:
    """Remove ANSI colour / formatting codes from text."""
    return _ANSI_RE.sub("", text)


def extract_traceback(stderr_tail: str) -> str:
    """Extract only the Traceback portion from stderr_tail, strip ANSI codes."""
    text = clean_ansi(stderr_tail)
    lines = text.split("\n")
    tb_start = None
    for i, line in enumerate(lines):
        if "Traceback (most recent call last):" in line:
            tb_start = i
    if tb_start is not None:
        return "\n".join(lines[tb_start:]).strip()
    return "\n".join(lines[-5:]).strip()


def extract_error_line(stderr_tail: str) -> str:
    """Extract just the final error line (e.g. 'ValueError: ...')."""
    text = clean_ansi(stderr_tail)
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    for line in reversed(lines):
        if re.match(r"^\w*(Error|Exception):", line):
            return line
    return lines[-1] if lines else ""


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# VERSION-AWARE RAG RETRIEVAL  (query = error text)
# ══════════════════════════════════════════════════════════════════════════════

def _acceptable_versions(library: str) -> list[str]:
    available = _CHROMA_VERSIONS.get(library, set())
    if not available:
        return []
    accept = set()
    best = VENV_TO_CHROMA.get(library)
    if best:
        accept.add(best)
    if "latest" in available:
        accept.add("latest")
    for v in available:
        if v.endswith("x"):
            accept.add(v)
    return sorted(accept)


def _query_library(lib: str, query_text: str, n_results: int) -> list[tuple[str, dict]]:
    """Query ChromaDB for a single library with version-aware filtering."""
    versions = _acceptable_versions(lib)
    if len(versions) == 1:
        where = {"$and": [{"library": lib}, {"version": versions[0]}]}
    elif len(versions) > 1:
        where = {"$and": [{"library": lib}, {"version": {"$in": versions}}]}
    else:
        where = {"library": lib}
    try:
        results = collection.query(
            query_texts=[query_text], n_results=n_results, where=where,
        )
        return list(zip(results["documents"][0], results["metadatas"][0]))
    except Exception:
        try:
            results = collection.query(
                query_texts=[query_text], n_results=n_results, where={"library": lib},
            )
            return list(zip(results["documents"][0], results["metadatas"][0]))
        except Exception:
            return []


def retrieve_rag_docs(
    error_text: str,
    code: str,
    n_retrieve: int = N_RETRIEVE,
    n_rerank: int = N_RERANK,
    max_query_len: int = MAX_QUERY,
    max_ctx_chars: int = MAX_CTX_CHARS,
    min_score: float = MIN_RERANKER_SCORE,
) -> list[tuple[float, str, dict]]:
    """
    Error-based two-stage retrieval.  Returns list of (score, doc_text, metadata)
    for the top-k reranked docs that pass the score gate.
    """
    libs = detect_libraries(code)
    query_text = error_text[:max_query_len]
    known_libs = [lib for lib in libs if lib in _CHROMA_VERSIONS]
    if not known_libs:
        return []

    # Stage 1: per-library bi-encoder retrieval
    all_candidates = []
    seen_docs = set()
    for lib in known_libs:
        for doc, meta in _query_library(lib, query_text, n_retrieve):
            if doc not in seen_docs:
                all_candidates.append((doc, meta))
                seen_docs.add(doc)
    if not all_candidates:
        return []

    # Stage 2: cross-encoder reranking + score gate
    pairs = [(query_text, doc) for doc, _ in all_candidates]
    scores = reranker.predict(pairs)
    scored = sorted(zip(scores, all_candidates), key=lambda x: x[0], reverse=True)
    passing = [(s, doc, meta) for s, (doc, meta) in scored if s > min_score]
    if not passing:
        return []

    # Keep top-k, respect char budget
    result = []
    total_chars = 0
    for score, doc, meta in passing[:n_rerank]:
        if total_chars + len(doc) > max_ctx_chars and result:
            break
        result.append((float(score), doc, meta))
        total_chars += len(doc)
    return result


def format_raw_docs(docs: list[tuple[float, str, dict]]) -> str:
    """Format reranked docs into a single string (used as input to summariser)."""
    parts = []
    for score, doc, meta in docs:
        parts.append(f"[{meta['library']} v{meta['version']}  score={score:.2f}]\n{doc}")
    return "\n---\n".join(parts)

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# DOC SUMMARISATION  (small LLM → bullet-point hints)
# ══════════════════════════════════════════════════════════════════════════════

_SUMMARIZE_SYSTEM = (
    "You are a concise technical summariser.\n"
    "Given documentation snippets and an error message, produce a short list of "
    "actionable bullet-point hints that are directly relevant to fixing the error.\n"
    "Rules:\n"
    "- Return ONLY bullet points (lines starting with \"- \").\n"
    "- Maximum 5 bullets, each ≤ 1 sentence.\n"
    "- Focus on the API signature, correct parameter names, or usage pattern that fixes the error.\n"
    "- Do NOT include code blocks or examples.\n"
    "- If the docs are not relevant to the error, return: - No relevant hints found."
)


def summarise_docs(
    raw_docs_text: str,
    error_line: str,
) -> str:
    """
    Call the summariser LLM to condense raw doc chunks + error into
    concise bullet-point hints for the code-fixer.
    """
    if not raw_docs_text.strip():
        return ""

    user_msg = (
        f"Error: {error_line}\n\n"
        f"Documentation snippets:\n{raw_docs_text}"
    )

    messages = [
        SystemMessage(content=_SUMMARIZE_SYSTEM),
        HumanMessage(content=user_msg),
    ]

    try:
        content, _ = _call_llm(llm_summarizer, messages)
        # Keep only lines that look like bullets
        bullets = [
            line.strip()
            for line in (content or "").split("\n")
            if line.strip().startswith("- ")
        ]
        return "\n".join(bullets) if bullets else content.strip()
    except Exception as e:
        print(f"  [WARN] summariser failed: {e}")
        return ""  # graceful degradation: skip hints


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SYSTEM & USER PROMPTS  (category-aware)
# ══════════════════════════════════════════════════════════════════════════════

CODER_SYSTEM_PROMPT = (
    "You fix Python programs.\n"
    "Return EXACTLY one block wrapped like this and nothing else:\n"
    "<correct_code>\n"
    "(python code only)\n"
    "</correct_code>\n"
    "Rules:\n"
    "- Do NOT echo the prompt or input.\n"
    "- Do NOT use markdown or backticks.\n"
    "- The python code MUST NOT contain the characters '<' or '>' anywhere.\n"
    "- Output exactly one opening and one closing tag."
)

def build_user_prompt(
    failed_code: str,
    traceback_text: str,
    bullet_hints: str,
    category: str,
) -> str:
    """
    Build the user prompt for the code-fixer LLM.
    Different templates for each error category:
      syntax_error  → code only, ask to fix syntax
      name_error    → code + traceback, ask to fix naming
      timeout       → code only, best-effort
      rag_error     → code + traceback + (optional) bullet hints
    """
    if category == "syntax_error":
        return (
            "Fix this Python file.\n"
            "Return the full corrected python file inside <correct_code> tags.\n\n"
            "Error: SyntaxError - the code failed to parse.\n\n"
            f"Incorrect file:\n{failed_code}"
        )

    if category == "name_error":
        return (
            "Fix this Python file.\n"
            "Return the full corrected python file inside <correct_code> tags.\n\n"
            "Traceback tail:\n"
            f"{traceback_text}\n\n"
            "Incorrect file:\n"
            f"{failed_code}\n"
            )

    if category == "timeout":
        return (
            "Fix this Python file.\n"
            "Return the full corrected python file inside <correct_code> tags.\n\n"
            "Error: Timeout - the code took too long to execute. Fix performance issues or infinite loops.\n\n"
            f"Incorrect file:\n{failed_code}"
        )

    # rag_error — full prompt with optional bullet hints
    prompt = (
        "Fix this Python file.\n"
        "Return the full corrected python file inside <correct_code> tags.\n\n"
        "Traceback tail:\n"
        f"{traceback_text}\n\n"
        "Incorrect file:\n"
        f"{failed_code}\n"
)
    if bullet_hints.strip():
        prompt += f"\n\nHints from documentation:\n{bullet_hints}"
    return prompt

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# RESPONSE PARSING
# ══════════════════════════════════════════════════════════════════════════════

# FIM markers that indicate the model went off-track
FIM_MARKERS = ("<|fim_middle|>", "<|fim_prefix|>", "<|fim_suffix|>", "<|file_sep|>")


def _cut_at_fim(text: str) -> str:
    """Cut text at the first FIM marker to avoid garbage after the code."""
    text = text or ""
    cut = len(text)
    for m in FIM_MARKERS:
        j = text.find(m)
        if j != -1:
            cut = min(cut, j)
    return text[:cut].strip()


def _strip_prompt_echo(text: str) -> str:
    """
    If the model echoed the prompt at the start, strip it.
    Detect by looking for 'Fix this Python file' at the very start.
    """
    text = (text or "").strip()
    # If output starts with the prompt, try to find actual code after
    if text.startswith("Fix this Python file"):
        # Find the first <correct_code> tag
        idx = text.find("<correct_code>")
        if idx != -1:
            text = text[idx:]
    return text


def extract_tag(text: str, tag: str) -> str:
    text = text or ""
    m = re.search(rf"<{tag}>\s*(.*?)\s*</{tag}>", text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else ""


def strip_code_fences(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"^\s*```[a-zA-Z0-9_-]*\s*", "", s)
    s = re.sub(r"\s*```\s*$", "", s)
    return s.strip()


def extract_correct_code_and_error(raw: str):
    """Parse the model output to extract corrected code and error type."""
    # Step 1: Cut at FIM markers
    raw = _cut_at_fim(raw or "")
    
    # Step 2: Strip prompt echo if present
    raw = _strip_prompt_echo(raw)
    
    # Step 3: Extract <correct_code> tag
    code = extract_tag(raw, "correct_code")
    
    # Fallback: look for ```python ... ``` block
    if not code.strip():
        m = re.search(r"```python\s*(.*?)```", raw, flags=re.DOTALL | re.IGNORECASE)
        if m:
            code = m.group(1)
    
    # Fallback: any fenced ``` ... ``` block
    if not code.strip():
        m = re.search(r"```\s*(.*?)```", raw, flags=re.DOTALL)
        if m:
            code = m.group(1)
    
    code = strip_code_fences(code or "")
    err = extract_tag(raw, "error_type").strip()
    return code, err

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# UTILITY HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def similarity_ratio(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()


def safe_sample_id(idx: int, title: str) -> str:
    safe = re.sub(r"[^a-zA-Z0-9_]", "_", title)[:40]
    return f"{idx:03d}_{safe}"


def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p


print("All helpers ready (library detection, RAG retrieval, summarisation, category-aware prompts, parsing).")

All helpers ready (library detection, RAG retrieval, summarisation, category-aware prompts, parsing).


## 7 — Run Error-Based RAG Pipeline (All 21 Failed Samples)

For each failed sample, branch by error category:

| Category | Steps |
|----------|-------|
| **syntax_error** | Prompt code-fixer with code only → *"fix the syntax error"* |
| **name_error** | Prompt code-fixer with code + traceback → *"fix the naming error"* |
| **timeout** | Prompt code-fixer with code only → *"fix performance issues"* |
| **rag_error** | Retrieve 10 docs → rerank 3 → summarise → prompt with code + traceback + bullet hints |

In [ ]:
#===========================================================
#6. RAG Pipeline – Process each "failed" sample
#===========================================================
summary: list[dict] = []

# ── warm-up call ──
print("Warming up model with a dummy call...")
_warmup_msg = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Say OK"),
]
_t0_warm = time.time()
_ = llm_coder.invoke(_warmup_msg)
print(f"  Warm-up done (latency={time.time() - _t0_warm:.1f}s)\n")

N_SAMPLES = len(eval_examples)
for i, entry in enumerate(tqdm(failed_entries[:N_SAMPLES], desc="RAG Pipeline")):
    smoke_idx = entry["idx"]
    category = entry["error_category"]
    title = entry["eval_sample"]["title"]
    
    # Skip entries with invalid input code
    if not entry.get("valid_input", True):
        print(f"  Skipping idx={smoke_idx} — invalid input code")
        continue

    sid = f"{smoke_idx:03d}_{title.replace(' ', '_')[:40]}"
    sample_dir = ensure_dir(OUT_DIR / sid)

    try:
        failed_code = entry["prediction_code"]
        tb_text = entry.get("traceback", "") or ""
        stderr = entry.get("stderr_tail", "") or ""
        correct = entry["eval_sample"].get("correct_code", "") or ""

        # ── decide if RAG is needed ──
        rag_used = False
        raw_docs_text = ""
        bullet_hints = ""

        if category.lower() in ("rag_error", "library_error", "runtime_error"):
            # use error as query for RAG
            query_text = (tb_text[:MAX_QUERY] if tb_text.strip() else stderr[:MAX_QUERY])
            docs = retrieve_rag_docs(query_text, failed_code)
            if docs:
                rag_used = True
                raw_docs_text = format_raw_docs(docs)
                error_line = extract_error_line(stderr)
                bullet_hints = summarise_docs(raw_docs_text, error_line)

        # ── build prompt ──
        traceback_text = extract_traceback(stderr) if stderr else ""
        user_prompt = build_user_prompt(
            failed_code=failed_code,
            traceback_text=traceback_text,
            bullet_hints=bullet_hints,
            category=category,
        )

        # ── call the code-fixing LLM ──
        messages = [
            SystemMessage(content=CODER_SYSTEM_PROMPT),
            HumanMessage(content=user_prompt),
        ]
        ts = time.time()
        raw_output = llm_coder.invoke(messages).content
        latency = time.time() - ts

        # ── parse response ──
        pred_code, pred_err = extract_correct_code_and_error(raw_output or "")
        status = "OK" if pred_code.strip() else "UNPARSEABLE_OUTPUT"

        # ── calculate similarity to reference ──
        sim_to_ref = similarity_ratio(pred_code, correct) if pred_code.strip() and correct.strip() else 0.0

        # ── save artifacts ──
        (sample_dir / "prompt.txt").write_text(user_prompt, encoding="utf-8")
        (sample_dir / "raw_model_output.txt").write_text(raw_output or "", encoding="utf-8")
        (sample_dir / "prediction.py").write_text(pred_code or "", encoding="utf-8")
        (sample_dir / "predicted_error_type.txt").write_text(pred_err or "", encoding="utf-8")
        (sample_dir / "failed_code.py").write_text(failed_code or "", encoding="utf-8")
        if raw_docs_text:
            (sample_dir / "rag_docs_raw.txt").write_text(raw_docs_text, encoding="utf-8")
        if bullet_hints:
            (sample_dir / "bullet_hints.txt").write_text(bullet_hints, encoding="utf-8")
        if traceback_text:
            (sample_dir / "traceback.txt").write_text(traceback_text, encoding="utf-8")
        if stderr:
            (sample_dir / "stderr_tail.txt").write_text(stderr, encoding="utf-8")
        if correct:
            (sample_dir / "correct.py").write_text(correct, encoding="utf-8")

        summary.append({
            "idx": smoke_idx,
            "title": title,
            "category": category,
            "status": status,
            "rag_used": rag_used,
            "hints_len": len(bullet_hints),
            "latency_s": round(latency, 2),
            "pred_error": (pred_err or "")[:80],
            "sim_to_ref": round(sim_to_ref, 4),
        })

        print(
            f"  [{i+1}/{min(N_SAMPLES, len(failed_entries))}] idx={smoke_idx} "
            f"[{category:13s}] status={status} rag={rag_used} sim={sim_to_ref:.2%} "
            f"hints={len(bullet_hints)}ch latency={latency:.1f}s | {(pred_err or '')[:60]}"
        )

    except Exception as e:
        summary.append({
            "idx": smoke_idx,
            "title": title,
            "category": category,
            "status": "EXCEPTION",
            "error": repr(e),
            "sim_to_ref": 0.0,
        })
        (sample_dir / "GEN_EXCEPTION.txt").write_text(repr(e), encoding="utf-8")
        print(f"  [{i+1}] idx={smoke_idx} EXCEPTION: {e}")

# ── write summary JSON ──
summary_path = OUT_DIR / "summary_rag_baseline.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Pipeline complete. {len(summary)} samples processed.")
#print(f"  Outputs saved to: {OUT_DIR}")
print(f"  Summary saved to: {summary_path.name}")

Warming up model with a dummy call...
  Warm-up done (latency=0.7s)



RAG Pipeline:   5%|▍         | 1/21 [00:08<02:43,  8.19s/it]

  [1/21] idx=2 [syntax_error ] status=UNPARSEABLE_OUTPUT rag=False hints=0ch latency=8.2s | 


RAG Pipeline:  10%|▉         | 2/21 [00:11<01:35,  5.05s/it]

  [2/21] idx=11 [syntax_error ] status=OK rag=False hints=0ch latency=2.8s | 


...

  [19/21] idx=84 [rag_error    ] status=OK rag=True hints=514ch latency=4.8s | 


RAG Pipeline:  95%|█████████▌| 20/21 [07:57<00:28, 28.60s/it]

  [20/21] idx=87 [rag_error    ] status=OK rag=True hints=26ch latency=7.3s | 
  [WARN] attempt 1/6 failed (RateLimitError): Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'openai/gpt-oss-20b:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'OpenInference', 'is_byok': False}}, 'user_id': 'user_34T64NRW09AE1IOhHpOs4qHu0F8'}
         retrying in 1.0s ...


RAG Pipeline: 100%|██████████| 21/21 [08:28<00:00, 24.23s/it]

  [21/21] idx=88 [rag_error    ] status=UNPARSEABLE_OUTPUT rag=True hints=26ch latency=7.1s | 

✓ Pipeline complete. 21 samples processed.
  Summary saved to: summary_rag_baseline.json


## 9 — Results Summary

In [22]:
status_counts = Counter(row["status"] for row in summary)
cat_counts = Counter(row["category"] for row in summary)
latencies = [r["latency_s"] for r in summary if r.get("latency_s") is not None]
rag_used_cnt = sum(1 for r in summary if r.get("rag_used"))
rag_total = sum(1 for r in summary if r.get("category") == "rag_error")
hint_lens = [r["hints_len"] for r in summary if r.get("hints_len") is not None and r["hints_len"] > 0]
#similarities = [r["sim_to_ref"] for r in summary if r.get("sim_to_ref") is not None]

print("=" * 85)
print(f"Baseline RAG Pipeline Summary  ({len(summary)} failed samples)")
print("=" * 85)

print(f"\nStatus:")
for st, cnt in status_counts.most_common():
    print(f"  {st:25s}: {cnt:4d}  ({cnt/len(summary)*100:.1f}%)")

print(f"\nCategory:")
for cat in ["syntax_error", "name_error", "rag_error", "timeout"]:
    cnt = cat_counts.get(cat, 0)
    print(f"  {cat:15s}: {cnt:4d}")

print(f"\nRAG usage: {rag_used_cnt}/{rag_total} rag_error samples received RAG docs + bullet hints")

if latencies:
    print(f"\nCode-fixer latency (s):")
    print(f"  Mean={sum(latencies)/len(latencies):.2f}  "
          f"Min={min(latencies):.2f}  Max={max(latencies):.2f}")

if hint_lens:
    print(f"\nBullet hints length (chars, rag_error only):")
    print(f"  Mean={sum(hint_lens)/len(hint_lens):.0f}  "
          f"Min={min(hint_lens)}  Max={max(hint_lens)}")

#if similarities:
#    print(f"\nSimilarity to reference (pred vs correct_code):")
 #   print(f"  Mean={sum(similarities)/len(similarities):.2%}  "
  #        f"Min={min(similarities):.2%}  Max={max(similarities):.2%}")
    # high_sim = sum(1 for s in similarities if s >= 0.8)
    # med_sim = sum(1 for s in similarities if 0.5 <= s < 0.8)
    # low_sim = sum(1 for s in similarities if s < 0.5)
    # print(f"  ≥80%: {high_sim}   50-80%: {med_sim}   <50%: {low_sim}")

# Per-sample detail table
print(f"\nPer-sample detail:")
header = f"{'idx':>4}  {'Category':<14}  {'Title':<35}  {'Status':<18}  {'RAG':>4}  {'Sim':>6}  {'Hints':>6}"
print(header)
print("-" * len(header))
for r in summary:
    rag_str = "yes" if r.get("rag_used") else " — " if r["category"] != "rag_error" else "no"
    sim_pct = f"{r.get('sim_to_ref', 0)*100:.1f}%" if r.get("sim_to_ref") is not None else "  —  "
    print(
        f"{r['idx']:4d}  {r['category']:<14}  {r['title'][:35]:<35}  {r['status']:<18}  "
        f"{rag_str:>4}  {sim_pct:>6}  {r.get('hints_len', 0):>6}"
    )

Baseline RAG Pipeline Summary  (21 failed samples)

Status:
  OK                       :   17  (81.0%)
  UNPARSEABLE_OUTPUT       :    4  (19.0%)

Category:
  syntax_error   :    4
  name_error     :    2
  rag_error      :   14
  timeout        :    1

RAG usage: 14/14 rag_error samples received RAG docs + bullet hints

Code-fixer latency (s):
  Mean=5.46  Min=2.52  Max=8.48

Bullet hints length (chars, rag_error only):
  Mean=205  Min=26  Max=767

Per-sample detail:
 idx  Category        Title                                Status               RAG     Sim   Hints
---------------------------------------------------------------------------------------------------
   2  syntax_error    CIFAR-10 Deep CNN Data Augmentation  UNPARSEABLE_OUTPUT    —      —         0
  11  syntax_error    MNIST Digit Distribution Check       OK                    —      —         0
  13  syntax_error    Fashion MNIST CNN Early Stopping     OK                    —      —         0
  14  name_error      Digit

## 10 — Evaluate RAG Outputs (Syntax + Runtime)

Now we evaluate the RAG-generated predictions:
1. **Syntax check** — `py_compile` to verify the code parses correctly
2. **Patch for fast eval** — Cap epochs, disable plt.show(), shrink datasets
3. **Runtime test** — Execute in subprocess with timeout

Only samples with `status=OK` (i.e., parseable output) are evaluated.

In [39]:
import py_compile
import subprocess
import sys

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
TIMEOUT = 300       # seconds per script
FORCE_CPU = False   # set True to disable GPU

RUNTIME_DIR = OUT_DIR / "runtime_eval"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output dir    : {OUT_DIR}")
print(f"Runtime dir   : {RUNTIME_DIR}")

Output dir    : C:\Users\hbahmanyar\MentorApp\RAG_Pipelines\RAG_outputs\RAG_with_Baseline
Runtime dir   : C:\Users\hbahmanyar\MentorApp\RAG_Pipelines\RAG_outputs\RAG_with_Baseline\runtime_eval


In [40]:
# ══════════════════════════════════════════════════════════════════════════════
# SYNTAX CHECK (py_compile)
# ══════════════════════════════════════════════════════════════════════════════

def syntax_check(pred_files, out_path):
    """
    Compile-check Python files and write a JSON report.
    Returns list of result dicts: idx, file, ok, seconds, error
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    syntax_results = []
    for i, f in enumerate(pred_files, 1):
        f = Path(f)
        t0 = time.time()
        try:
            py_compile.compile(str(f), doraise=True)
            ok = True
            err = ""
        except Exception as e:
            ok = False
            err = repr(e)

        syntax_results.append({
            "idx": i,
            "file": str(f),
            "ok": ok,
            "seconds": round(time.time() - t0, 4),
            "error": err,
        })

    out_path.write_text(json.dumps(syntax_results, indent=2), encoding="utf-8")

    correct = sum(r["ok"] for r in syntax_results)
    incorrect = len(syntax_results) - correct
    print(f"Syntax check: {correct} OK, {incorrect} failed")

    return syntax_results


print("✓ syntax_check() defined")

✓ syntax_check() defined


In [41]:
# ══════════════════════════════════════════════════════════════════════════════
# FAST EVAL PATCHING
# ══════════════════════════════════════════════════════════════════════════════

def patch_fast_eval(code: str, skip_epoch_patch: bool = False) -> str:
    """Patch code to run faster during evaluation."""
    header = r'''
import os
FAST_EVAL = os.environ.get("FAST_EVAL", "0") == "1"
if FAST_EVAL:
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
'''
    code = header + "\n" + code

    # Cap epochs (unless skip_epoch_patch is True)
    if not skip_epoch_patch:
        code = re.sub(r"(epochs\s*=\s*)\d+", r"\g<1>5", code)
        code = re.sub(r"(fit\([^)]*?\bepochs\s*=\s*)\d+", r"\g<1>5", code, flags=re.DOTALL)

    # Cap steps
    code = re.sub(r"(steps_per_epoch\s*=\s*)\d+", r"\g<1>1", code)
    code = re.sub(r"(validation_steps\s*=\s*)\d+", r"\g<1>1", code)

    # Disable plt.show
    code = re.sub(r"\bplt\.show\(\)", "print('[FAST_EVAL] plt.show() skipped')", code)

    # Shrink CIFAR if present
    shrink = r'''
if FAST_EVAL:
    try:
        x_train = x_train[:512]; y_train = y_train[:512]
        x_test  = x_test[:128]; y_test  = y_test[:128]
    except Exception:
        pass
'''
    code = re.sub(
        r"(=\s*cifar10\.load_data\(\)\s*)",
        r"\1\n" + shrink + "\n",
        code
    )

    # Replace fetch_california_housing -> load_diabetes
    code = re.sub(
        r"from\s+sklearn\.datasets\s+import\s+([^\n]*?)\bfetch_california_housing\b([^\n]*)",
        lambda m: f"from sklearn.datasets import {m.group(1).strip().rstrip(', ')}"
                  + (", " if m.group(1).strip() else "")
                  + "load_diabetes"
                  + ((", " + m.group(2).strip().lstrip(", ")) if m.group(2).strip() else ""),
        code
    )
    code = re.sub(r"\bfetch_california_housing\b", "load_diabetes", code)

    return code


def materialize_patched(src: Path, work: Path, root: Path, skip_epoch_patch: bool = False) -> Path:
    """Write patched version of src to work directory."""
    rel = src.relative_to(root)
    dst = work / rel
    dst.parent.mkdir(parents=True, exist_ok=True)

    code = src.read_text(encoding="utf-8", errors="ignore")
    dst.write_text(patch_fast_eval(code, skip_epoch_patch=skip_epoch_patch), encoding="utf-8")
    return dst


print("✓ patch_fast_eval() and materialize_patched() defined")

✓ patch_fast_eval() and materialize_patched() defined


In [42]:
# ══════════════════════════════════════════════════════════════════════════════
# SCRIPT RUNNER (subprocess with timeout)
# ══════════════════════════════════════════════════════════════════════════════

def run_script(path: Path, timeout=TIMEOUT):
    """Run a Python script in subprocess and capture output."""
    env = os.environ.copy()
    env["FAST_EVAL"] = "1"
    if FORCE_CPU:
        env["CUDA_VISIBLE_DEVICES"] = ""

    t0 = time.time()
    try:
        proc = subprocess.run(
            [sys.executable, str(path)],
            cwd=str(path.parent),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout
        )
        dt = time.time() - t0
        return {
            "ok": proc.returncode == 0,
            "returncode": proc.returncode,
            "seconds": round(dt, 3),
            "stdout_tail": "\n".join((proc.stdout or "").splitlines()[-30:]),
            "stderr_tail": "\n".join((proc.stderr or "").splitlines()[-60:]),
        }
    except subprocess.TimeoutExpired as e:
        dt = time.time() - t0
        out = e.stdout or ""
        err = e.stderr or ""
        return {
            "ok": False,
            "returncode": None,
            "seconds": round(dt, 3),
            "stdout_tail": "\n".join(out.splitlines()[-30:]) if out else "",
            "stderr_tail": "TIMEOUT\n" + ("\n".join(err.splitlines()[-60:]) if err else ""),
        }


print("✓ run_script() defined")

✓ run_script() defined


### 10.1 — Find RAG output prediction files

In [43]:
# Find all prediction.py files from RAG outputs
pred_files = sorted(OUT_DIR.rglob("prediction.py"))

# Filter to only non-empty files (skip unparseable outputs)
valid_pred_files = []
for f in pred_files:
    content = f.read_text(encoding="utf-8", errors="ignore").strip()
    if content and len(content) > 50:  # Skip empty/placeholder files
        valid_pred_files.append(f)

print(f"Total prediction.py files : {len(pred_files)}")
print(f"Non-empty prediction files: {len(valid_pred_files)}")

# Show the files
for i, f in enumerate(valid_pred_files, 1):
    sample_dir = f.parent.name
    size = len(f.read_text(encoding="utf-8", errors="ignore"))
    print(f"  {i:2d}. {sample_dir:<45s}  ({size:5d} chars)")

Total prediction.py files : 21
Non-empty prediction files: 17
   1. 011_MNIST_Digit_Distribution_Check             ( 1906 chars)
   2. 013_Fashion_MNIST_CNN_Early_Stopping           ( 3763 chars)
   3. 018_Reuters_News_Topic_Classification          ( 3129 chars)
   4. 027_Titanic_Survival_ROC_Curve                 ( 5122 chars)
   5. 032_California_Counterfactual_Explainability   ( 7889 chars)
   6. 034_Digits_Autoencoder_Reconstruction          ( 2711 chars)
   7. 047_Penguins_Migration_Time_Series_ARIMA       ( 4354 chars)
   8. 048_Heart_Model_Calibration_Plot               ( 2033 chars)
   9. 058_MNIST_Voting_Ensemble_Classification       ( 4368 chars)
  10. 059_Diabetes_Progression_Neural_Network_Regr   ( 4049 chars)
  11. 062_Digits_Classification_using_KNN            ( 2580 chars)
  12. 076_Breast_Cancer_Diagnosis_with_LightGBM      ( 3194 chars)
  13. 077_Fashion_MNIST_Transfer_Learning_with_Mob   ( 5069 chars)
  14. 078_Iris_tSNE_Cluster_Visualization            ( 2659 chars)


### 10.2 — Syntax Check (py_compile)

In [44]:
# Run syntax check on all valid prediction files
syntax_report_path = RUNTIME_DIR / "syntax_report.json"
syntax_results = syntax_check(valid_pred_files, syntax_report_path)

# Show failed files
failed_syntax = [r for r in syntax_results if not r["ok"]]
if failed_syntax:
    print(f"\nSyntax errors ({len(failed_syntax)}):")
    for r in failed_syntax:
        sample_dir = Path(r["file"]).parent.name
        print(f"  - {sample_dir}")
        print(f"    {r['error'][:100]}")
else:
    print("\n✓ All files passed syntax check!")

Syntax check: 16 OK, 1 failed

Syntax errors (1):
  - 013_Fashion_MNIST_CNN_Early_Stopping
    PyCompileError('  File "C:\\Users\\hbahmanyar\\MentorApp\\RAG_Pipelines\\RAG_outputs\\RAG_with_Basel


### 10.3 — Runtime Test (Smoke Test)

In [45]:
# Identify files that passed syntax check
syntax_ok = {r["file"] for r in syntax_results if r["ok"]}
print(f"Files passing syntax: {len(syntax_ok)}")

# Create patched directory
PATCHED_DIR = RUNTIME_DIR / "patched"
PATCHED_DIR.mkdir(parents=True, exist_ok=True)

# Samples that should NOT have epochs patched (run with original epochs)
NO_EPOCH_PATCH = {"081"}  # idx 81 has epochs=2 which is fine

# Run smoke test on all syntax-passing files
smoke_results = []
for i, src in enumerate(tqdm(valid_pred_files, desc="Smoke Test"), 1):
    sample_dir = src.parent.name
    sample_idx = sample_dir.split("_")[0]  # e.g., "081" from "081_Sample_Title"
    
    if str(src) not in syntax_ok:
        smoke_results.append({
            "idx": i,
            "sample": sample_dir,
            "file": str(src),
            "ok": False,
            "skipped": True,
            "reason": "syntax_failed"
        })
        continue

    # Patch and run
    try:
        patched = materialize_patched(
            src=src, 
            work=PATCHED_DIR, 
            root=OUT_DIR,
            skip_epoch_patch=(sample_idx in NO_EPOCH_PATCH)
        )
        r = run_script(patched)
        r.update({
            "idx": i,
            "sample": sample_dir,
            "file": str(src),
            "patched": str(patched),
            "skipped": False
        })
        smoke_results.append(r)
        
        # Print progress
        status = "✓" if r["ok"] else "✗"
        print(f"  {status} {sample_dir[:40]:<40s}  {r['seconds']:6.1f}s")
        
    except Exception as e:
        smoke_results.append({
            "idx": i,
            "sample": sample_dir,
            "file": str(src),
            "ok": False,
            "skipped": False,
            "error": repr(e)
        })
        print(f"  ✗ {sample_dir[:40]:<40s}  EXCEPTION: {e}")

# Save smoke report
smoke_report_path = RUNTIME_DIR / "smoke_report.json"
smoke_report_path.write_text(json.dumps(smoke_results, indent=2), encoding="utf-8")
print(f"\n✓ Smoke report saved to: {smoke_report_path.name}")

Files passing syntax: 16


Smoke Test:   6%|▌         | 1/17 [00:10<02:40, 10.05s/it]

  ✓ 011_MNIST_Digit_Distribution_Check          10.0s


Smoke Test:  18%|█▊        | 3/17 [00:19<01:24,  6.02s/it]

  ✗ 018_Reuters_News_Topic_Classification        9.2s


...

  ✓ 081_IMDB_Sentiment_Bidirectional_LSTM_Em   144.0s


Smoke Test:  94%|█████████▍| 16/17 [04:59<00:39, 39.25s/it]

  ✗ 084_Adult_Income_CatBoost_Classifier         5.9s


Smoke Test: 100%|██████████| 17/17 [05:12<00:00, 18.39s/it]

  ✗ 087_IMDB_Sentiment_BERT_Fine-Tuning         12.8s

✓ Smoke report saved to: smoke_report.json


### 10.4 — Evaluation Summary

In [46]:
# ══════════════════════════════════════════════════════════════════════════════
# EVALUATION SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

total = len(smoke_results)
passed = sum(1 for r in smoke_results if r.get("ok") is True and not r.get("skipped", False))
failed = sum(1 for r in smoke_results if r.get("ok") is False and not r.get("skipped", False))
skipped = sum(1 for r in smoke_results if r.get("skipped", False))

# Percentages
pass_pct_total = (passed / total * 100) if total else 0.0
evaled = passed + failed
pass_pct_evaled = (passed / evaled * 100) if evaled else 0.0

print("=" * 80)
print("RAG Baseline — Runtime Evaluation Summary")
print("=" * 80)

print(f"\nTotal RAG outputs evaluated: {total}")
print(f"  Passed (runtime OK)     : {passed:3d}  ({pass_pct_total:.1f}%)")
print(f"  Failed (runtime error)  : {failed:3d}")
print(f"  Skipped (syntax error)  : {skipped:3d}")

if evaled:
    print(f"\nPass rate (excl. skipped): {pass_pct_evaled:.1f}% ({passed}/{evaled})")

# Runtime stats
runtimes = [r["seconds"] for r in smoke_results if r.get("seconds") and not r.get("skipped")]
if runtimes:
    print(f"\nRuntime (seconds):")
    print(f"  Mean: {sum(runtimes)/len(runtimes):.1f}s  Min: {min(runtimes):.1f}s  Max: {max(runtimes):.1f}s")

# Show failed samples
failed_samples = [r for r in smoke_results if not r.get("ok") and not r.get("skipped")]
if failed_samples:
    print(f"\nFailed samples ({len(failed_samples)}):")
    for r in failed_samples:
        stderr = r.get("stderr_tail", "")
        err_lines = [l.strip() for l in stderr.split("\n") if l.strip()]
        last_err = err_lines[-1] if err_lines else "unknown error"
        print(f"  - {r['sample'][:45]:<45s}  → {last_err[:60]}")

# Show passed samples
passed_samples = [r for r in smoke_results if r.get("ok")]
if passed_samples:
    print(f"\nPassed samples ({len(passed_samples)}):")
    for r in passed_samples:
        print(f"  ✓ {r['sample'][:50]:<50s}  ({r['seconds']:.1f}s)")

RAG Baseline — Runtime Evaluation Summary

Total RAG outputs evaluated: 17
  Passed (runtime OK)     :   8  (47.1%)
  Failed (runtime error)  :   8
  Skipped (syntax error)  :   1

Pass rate (excl. skipped): 50.0% (8/16)

Runtime (seconds):
  Mean: 19.5s  Min: 4.1s  Max: 144.0s

Failed samples (8):
  - 018_Reuters_News_Topic_Classification          → ValueError: Received an invalid value for `u
  - 034_Digits_Autoencoder_Reconstruction          → AssertionError: Reconstruction loss 0.040591
  - 048_Heart_Model_Calibration_Plot               → IndexError: boolean index did not match inde
  - 058_MNIST_Voting_Ensemble_Classification       → sklearn.exceptions.NotFittedError: This Voti
  - 077_Fashion_MNIST_Transfer_Learning_with_Mob   → tensorflow.python.framework.errors_impl.ResourceExhau
  - 078_Iris_tSNE_Cluster_Visualization            → ValueError: perplexity (200) must be less th
  - 084_Adult_Income_CatBoost_Classifier           → TypeError: Cannot setitem on a Categorical w
  - 0

In [48]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPARISON: RAG vs Original Pre_Test
# ══════════════════════════════════════════════════════════════════════════════

# Original Pre_Test results (88 eval samples)
total_eval_samples = 88
original_passed = 67
original_failed = 21
original_pass_rate = original_passed / total_eval_samples * 100

# RAG results on the 21 failed samples
rag_fixed = passed  # samples that now pass after RAG
rag_unparseable = len([r for r in summary if r.get("status") == "UNPARSEABLE_OUTPUT"])

# New totals after RAG
new_passed = original_passed + rag_fixed
new_pass_rate = new_passed / total_eval_samples * 100

print("=" * 80)
print("Overall Evaluation Summary")
print("=" * 80)

print(f"\n📊 Original Pre_Test (Baseline Qwen 2.5 Coder 7B):")
print(f"   Total eval samples    : {total_eval_samples}")
print(f"   Passed                : {original_passed}  ({original_pass_rate:.1f}%)")
print(f"   Failed                : {original_failed}  ({100 - original_pass_rate:.1f}%)")

print(f"\n🔧 RAG Pipeline Results (on {original_failed} failed samples):")
print(f"   Unparseable outputs   : {rag_unparseable}")
print(f"   Syntax errors         : {skipped}")
print(f"   Runtime errors        : {failed}")
print(f"   Runtime passed (FIXED): {rag_fixed}")

print(f"\n✅ After RAG Augmentation:")
print(f"   Total eval samples    : {total_eval_samples}")
print(f"   Passed                : {new_passed}  ({new_pass_rate:.1f}%)")
print(f"   Failed                : {total_eval_samples - new_passed}  ({100 - new_pass_rate:.1f}%)")

print(f"\n📈 Improvement:")
print(f"   Original pass rate    : {original_pass_rate:.1f}%  ({original_passed}/{total_eval_samples})")
print(f"   New pass rate         : {new_pass_rate:.1f}%  ({new_passed}/{total_eval_samples})")
print(f"   Improvement           : +{new_pass_rate - original_pass_rate:.1f}%  (+{rag_fixed} samples)")
print(f"   Fix rate (of failed)  : {rag_fixed}/{original_failed} = {rag_fixed/original_failed*100:.1f}%")

Overall Evaluation Summary

📊 Original Pre_Test (Baseline Qwen 2.5 Coder 7B):
   Total eval samples    : 88
   Passed                : 67  (76.1%)
   Failed                : 21  (23.9%)

🔧 RAG Pipeline Results (on 21 failed samples):
   Unparseable outputs   : 4
   Syntax errors         : 1
   Runtime errors        : 8
   Runtime passed (FIXED): 8

✅ After RAG Augmentation:
   Total eval samples    : 88
   Passed                : 75  (85.2%)
   Failed                : 13  (14.8%)

📈 Improvement:
   Original pass rate    : 76.1%  (67/88)
   New pass rate         : 85.2%  (75/88)
   Improvement           : +9.1%  (+8 samples)
   Fix rate (of failed)  : 8/21 = 38.1%
